# Lab 2 — One shot vs fix-it loop, plus sanity checks
The cheapest ways to catch what the AI gets wrong — better prompts, tests first, a fix-it loop, and sanity checks — and the surprise that every check has blind spots.

**Before you run:** upload the course pack (`aero20_s1.json`) and the captured response cache next to this notebook, or set `BASE_URL` in Step 2. Then *Runtime → Run all*: it runs with no API key and finishes in under 10 minutes on a free Colab CPU.

**How grading works, and its limits.** AI-written code runs in its own process and only computes outputs for the inputs it is sent; every comparison and pass/fail decision is made here, in the notebook, where that code cannot reach it. That stops AI code from faking a pass. It is **not** a security sandbox: the code could still read files or use the network, so use a clean runtime with no credentials, no mounted Drive and no confidential files.

**Step 1 — Install.** Colab already has numpy, scipy and pandas; this adds the few extras the lab uses.

In [ ]:
%pip install -q litellm datasets hypothesis pint  # pin versions after your first Colab run

**Step 2 — Config.** Leave `USE_CACHE = True` in class so results match the slides. `DOMAIN` switches editions once that edition's pack and cache exist (the quant edition's items are placeholders until then).

In [ ]:
DOMAIN = "aero"         # "aero" now; "quant" once its pack and cache exist
USE_CACHE = True        # False -> call a model live (if a live call fails, set True and rerun)
MODEL = "colab:"        # free model inside Colab; or a LiteLLM model string with your own key
TEMPERATURE = 0.8       # used by LiteLLM models; the free Colab model uses its own default
CACHE_MODEL = None      # which captured model to replay (None = as captured)
BASE_URL = ""           # course file host; leave "" if the files are uploaded to Colab
PACK_FILE = {"aero": "aero20_s1.json", "quant": "quant20_s1.json"}[DOMAIN]
CACHE_FILE = f"cache_s1_{DOMAIN}.json"
VARIANTS = ("minimal", "full", "tests_first")
ITEMS = {"aero": ["AE01", "AE03", "AE07", "AE09", "AE11", "AE12", "AE14", "AE15", "AE16", "AE19"], "quant": []}[DOMAIN]
AHA1 = {"aero": ("AE12", "D12"), "quant": None}[DOMAIN]          # test passes, sanity check fails
AHA2 = {"aero": ["D1", "D7"], "quant": []}[DOMAIN]               # sanity checks miss these
BREAKIT = {"aero": ["D1", "D7", "D12", "D5"], "quant": []}[DOMAIN]
PROPERTY = {"aero": ("AE01", 0.0, 10998.0, "f(x)[1] > f(x + 1.0)[1]"), "quant": None}[DOMAIN]
UNIT_DEMO = {"aero": ("AE01", 5, "kilometer", "meter"), "quant": None}[DOMAIN]

**Step 3 — Data** (collapsed). Loads the problem pack and the cached model responses, and defines the exact prompts sent.

In [ ]:
# @title Load the problem pack and the response cache { display-mode: "form" }
import json, os, hashlib, urllib.request
def _load(name):
    if not os.path.exists(name) and BASE_URL:
        try:
            urllib.request.urlretrieve(BASE_URL + name, name)
        except Exception:
            pass
    if not os.path.exists(name):
        raise FileNotFoundError(f"{name} not found: upload it next to this notebook, or set BASE_URL in Step 2.")
    return json.load(open(name))
PACK = _load(PACK_FILE)
PACK_FILE_SHA256 = hashlib.sha256(open(PACK_FILE, "rb").read()).hexdigest()   # hash of the file as distributed
CACHE = _load(CACHE_FILE) if USE_CACHE else {}
if USE_CACHE and CACHE_MODEL:   # replay ONE captured model; never substitute another
    CACHE = {k: dict(v, samples=[s for s in v["samples"] if s.get("model") == CACHE_MODEL]) for k, v in CACHE.items()}
    missing = sum(1 for v in CACHE.values() if not v["samples"])
    if missing:
        print(f"note: {missing} cached prompts have no response from {CACHE_MODEL}; those steps will say so rather than use another model")
FIXTURE = any(s.get("model", "").startswith("fixture:") for v in CACHE.values() for s in v["samples"])
if FIXTURE:
    print("WARNING: FIXTURE cache (the pack's own solutions and planted mistakes) - test data, not model output. Capture real responses before class.")
PROBLEMS = {p["id"]: p for p in PACK["problems"]}
DEFECTS = {d["id"]: d for d in PACK["defects"]}

WRAPPER = "Write the Python function below. Reply with the code only, in one ```python block."
def make_prompt(p, variant):
    """The exact prompt sent: the same one-line WRAPPER, then the task text for the variant."""
    text = p["prompt_minimal"] if variant == "minimal" else p["prompt"]
    if variant == "tests_first":
        text += "\nIt must pass these tests:\n" + "\n".join(
            f"- {p['entry_point']}({', '.join(map(repr, t['args']))}) returns {t['expected']!r} (relative tolerance {t['rel_tol']:g}"
            + (f", absolute tolerance {t['abs_tol']:g}" if t.get("abs_tol") else "") + ")"
            for t in p["tests"])
    return WRAPPER + "\n\n" + text

def he_prompt(r):
    return "Complete this Python function. Reply with the whole function in one ```python block.\n\n" + r["prompt"]

def feedback(p, r):
    if r.get("error"):
        return f"It failed with an error: {r['error']}."
    bad = [(t, x) for t, x in zip(p["tests"], r["tests"]) if not x["ok"]]
    if bad:
        t, x = bad[0]
        return f"For inputs {t['args']} it returned {x['value']}, but the expected value is {t['expected']}."
    return "It did not pass its tests."

def fix_prompt(p, code, r):
    return (make_prompt(p, "minimal") + "\n\nYour previous code:\n```python\n" + code + "```\n"
            + feedback(p, r) + " Return the corrected function only, in one ```python block.")

def check_property(code, item, lo, hi, rule, n=300):
    """Hypothesis runs HERE; the AI's function runs in its own process and only returns values."""
    try:
        from hypothesis import given, settings, strategies as st
    except ImportError:
        return "not installed - run Step 1"
    try:
        with RemoteFunction(PACK["imports"] + "\n" + code, PROBLEMS[item]["entry_point"], timeout=120) as f:
            @settings(max_examples=n, deadline=None)
            @given(st.floats(lo, hi))
            def prop(x):
                assert eval(rule, {"f": f, "x": x})
            prop()
        return f"the property held on every input it generated (up to {n}) - evidence, not proof"
    except Exception as e:
        return f"property FAILED - {type(e).__name__}: {str(e)[:100]}"

PROMPT_LOG = []                     # every prompt sent, with its fingerprint (exported by the labs)
def log_prompt(key, prompt):
    PROMPT_LOG.append({"key": key, "sha256_12": hashlib.sha256(prompt.encode()).hexdigest()[:12], "prompt": prompt})

def report_variants(items, variants, rows, results):
    """Two grids (visible tests, held-out cases) and per-variant scores from real captures only."""
    print("Visible reference tests:")
    show_grid(items, list(variants), lambda i, v: "pass" if results[i, v]["passed"] else "FAIL")
    print("Held-out cases (never shown to the model):")
    show_grid(items, list(variants), lambda i, v: "pass" if results[i, v]["holdout_passed"] else ("n/a" if results[i, v]["holdout_passed"] is None else "FAIL"))
    for v in variants:
        ev = [r for r in rows if r["variant"] == v and is_evidence(r["source"])]
        if not ev:
            print(f"{v:12s} no model evidence (fixture or mismatched captures) - no score computed"); continue
        k = sum(r["passed"] for r in ev); hv = [r for r in ev if r["holdout_passed"] is not None]
        hp = sum(r["holdout_passed"] for r in hv); hc = sum(r["holdout_cases_passed"] for r in hv); ht = sum(r["holdout_cases_total"] for r in hv)
        lo, hi = wilson(hc, ht) if ht else (0, 0)
        print(f"{v:12s} visible tests {k}/{len(ev)} | " + (f"held-out: {hp}/{len(hv)} problems pass both; {hc}/{ht} cases (95% interval {lo:.0%} to {hi:.0%})" if hv else "held-out: none"))
    print("sources:", sorted({r["source"] for r in rows}))

def run_repairs(items, codes, results, rounds=3):
    """Fix-it loop with a per-repair audit trail. A cached repair is used only if it was generated from
    exactly this repair prompt; otherwise it is recorded as stale and not attempted."""
    current = {i: (codes[i, "minimal"], results[i, "minimal"]) for i in items}
    by_round = [{"round": 0, "n": len(items), "attempted": 0, "missing_cache": 0, "stale_cache": 0, "new_passes": 0,
                 "passed": sum(r["passed"] for _, r in current.values())}]
    audit = []
    for rnd in range(1, rounds + 1):
        tally = {"attempted": 0, "missing_cache": 0, "stale_cache": 0}
        for item, (code, r) in [(i, cr) for i, cr in current.items() if not cr[1]["passed"]]:
            key, prompt = f"{item}|minimal|fix{rnd}", fix_prompt(PROBLEMS[item], code, r)
            src = provenance(key, prompt) if (not USE_CACHE or CACHE.get(key, {}).get("samples")) else "missing"
            status = "missing_cache" if src == "missing" else ("stale_cache" if "(prompt changed)" in src else "attempted")
            tally[status] += 1
            row = {"item": item, "round": rnd, "status": status, "source": src, "prompt_sha256_12": hashlib.sha256(prompt.encode()).hexdigest()[:12],
                   "previous_code_sha256_12": hashlib.sha256(code.encode()).hexdigest()[:12], "passed": None}
            if status == "attempted":
                log_prompt(key, prompt)
                new = extract_code(ask_llm(prompt, key)[0]); current[item] = (new, evaluate(new, PROBLEMS[item])); row["passed"] = current[item][1]["passed"]
            audit.append(row)
        passed = sum(r["passed"] for _, r in current.values())
        by_round.append({"round": rnd, "n": len(items), **tally, "new_passes": passed - by_round[-1]["passed"], "passed": passed})
    for b in by_round:
        b["pass_fraction"] = round(b["passed"] / b["n"], 3)
    return by_round, audit

print(PACK["pack"], PACK["version"], "|", len(PROBLEMS), "problems |", len(CACHE), "cached prompts | pack file sha256", PACK_FILE_SHA256[:12])


**Step 4 — Helpers** (collapsed). The same helper cell as every lab: model calls, a separate process for AI-written code, error bars, provenance, and result files.

In [ ]:
# @title Helpers — run this cell (no need to read it) { display-mode: "form" }
# Standard helpers for every lab notebook in the course. AI-written code runs in its own
# process and only computes outputs; every pass/fail decision is made in this notebook.
# Fault containment, NOT a security sandbox: use a clean runtime (no secrets, no mounted Drive). Identical in every notebook,
# inlined (not imported) so each notebook stands alone. Domain-free: the pack carries
# the domain. Relies on PACK, CACHE, USE_CACHE, MODEL from the config cell.
import json, re, sys, math, hashlib, subprocess, tempfile, textwrap, datetime, platform

def prompt_sha(text):
    return hashlib.sha256(text.encode()).hexdigest()[:12]

def ask_llm(prompt, key, n=1, temperature=None):
    """Return n completions. Cached by default (deterministic class runs); live otherwise."""
    if USE_CACHE:
        entry = CACHE.get(key)
        if entry is None or not entry.get("samples"):
            raise KeyError(f"no cached response for {key}" + (f" from CACHE_MODEL={globals().get('CACHE_MODEL')!r}" if globals().get("CACHE_MODEL") else "")
                           + "; set USE_CACHE = False to call a model")
        want = prompt_sha(prompt)
        matched = [s for s in entry["samples"] if s.get("prompt_sha") == want]
        samples = matched or entry["samples"]
        entry_sha = entry.get("prompt_sha")
        sample_shas = {s.get("prompt_sha") for s in samples if s.get("prompt_sha")}
        if not matched and ((sample_shas and want not in sample_shas) or (entry_sha and entry_sha != want)):
            print(f"note: prompt for {key} changed since it was cached - this response is NOT counted as evidence")
        return [s["text"] for s in samples[:n]]
    temperature = globals().get("TEMPERATURE", 0.8) if temperature is None else temperature
    out = []
    for _ in range(n):
        if MODEL.startswith("colab:"):                      # free in Colab, no key
            from google.colab import ai                     # verify the API each term
            name = MODEL.split(":", 1)[1] or None
            try:
                out.append(ai.generate_text(prompt, model_name=name) if name else ai.generate_text(prompt))
            except TypeError:
                out.append(ai.generate_text(prompt))
        else:                                               # any provider via LiteLLM + your key
            import litellm
            r = litellm.completion(model=MODEL, temperature=temperature,
                                   messages=[{"role": "user", "content": prompt}])
            out.append(r.choices[0].message.content)
    return out

def provenance(key, prompt=None):
    """Where a response came from: 'fixture' (test data, never evidence), 'capture:<model>', or 'live:<model>'.
    Adds '(prompt changed)' if the cached response was generated from a different prompt."""
    if not USE_CACHE:
        return f"live:{MODEL}"
    entry = CACHE.get(key, {}); samples = entry.get("samples") or [{}]
    want = prompt_sha(prompt) if prompt is not None else None
    chosen = next((s for s in samples if s.get("prompt_sha") == want), samples[0]) if want else samples[0]
    m = chosen.get("model", "unknown")
    tag = "fixture" if m.startswith("fixture:") else f"capture:{m}"
    if want is not None:
        own = chosen.get("prompt_sha", entry.get("prompt_sha"))
        if own and own != want:
            tag += " (prompt changed)"
    return tag

def extract_code(text):
    """First fenced python block, else the whole text."""
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip() + "\n"

# ---------------------------------------------------------------- grading: trust boundary
# The AI's code runs in its OWN process and only computes outputs for inputs it is sent.
# Every comparison, sanity check and pass/fail decision is made here, in the notebook, so the
# AI's code cannot fake a printed result or patch the grader's arithmetic. Values come back as
# plain JSON over a private pipe (not stdout) and are parsed safely. This is fault containment,
# NOT a security sandbox: the child can still read files (including the pack's reference values),
# inspect its parent via the OS, or use the network - so use a clean runtime with no secrets.
import os as _os, time as _time, selectors as _selectors, secrets as _secrets, signal as _signal
for _m in ("pandas", "csv", "hypothesis"):            # import before any AI code runs (no file shadowing)
    try:
        __import__(_m)
    except ImportError:
        pass

_CHILD = r"""
import sys, os, json, math
rfd, wfd, code_path, entry = int(sys.argv[1]), int(sys.argv[2]), sys.argv[3], sys.argv[4]
rin, wout = os.fdopen(rfd, "r"), os.fdopen(wfd, "w")
def enc(v):
    if hasattr(v, "tolist"): v = v.tolist()
    elif hasattr(v, "item") and not isinstance(v, (str, bytes)): v = v.item()
    if isinstance(v, bool) or v is None or isinstance(v, (int, str)): return v
    if isinstance(v, float): return v if math.isfinite(v) else {"__float__": repr(v)}
    if isinstance(v, complex): return {"__complex__": [v.real, v.imag]}
    if isinstance(v, tuple): return {"__tuple__": [enc(x) for x in v]}
    if isinstance(v, (set, frozenset)): return {"__set__": [enc(x) for x in v]}
    if isinstance(v, list): return [enc(x) for x in v]
    if isinstance(v, dict): return {"__dict__": [[enc(k), enc(x)] for k, x in v.items()]}
    return {"__repr__": repr(v)[:200]}
def dec(v):
    if isinstance(v, list): return [dec(x) for x in v]
    if isinstance(v, dict):
        if "__tuple__" in v: return tuple(dec(x) for x in v["__tuple__"])
        if "__dict__" in v: return {dec(k): dec(x) for k, x in v["__dict__"]}
    return v
def send(obj):
    wout.write(json.dumps(obj) + "\n"); wout.flush()
ns = {}
try:
    exec(compile(open(code_path).read(), "<candidate>", "exec"), ns)
    f = ns[entry]
    send({"ready": True})
except BaseException as e:
    send({"load_error": f"{type(e).__name__}: {e}"[:200]}); sys.exit(0)
for line in rin:
    req = json.loads(line)
    try:
        send({"id": req["id"], "value": enc(f(*dec(req["args"]), **dec(req.get("kwargs", {}))))})
    except BaseException as e:
        send({"id": req["id"], "error": f"{type(e).__name__}: {e}"[:200]})
"""

class CandidateError(Exception):
    """The AI-written code raised, failed to load, broke the protocol, or timed out."""

def _dec(v):
    if isinstance(v, list):
        return [_dec(x) for x in v]
    if isinstance(v, dict):
        if "__tuple__" in v: return tuple(_dec(x) for x in v["__tuple__"])
        if "__set__" in v: return set(_dec(x) for x in v["__set__"])
        if "__float__" in v: return float(v["__float__"]) if v["__float__"] in ("inf", "-inf", "nan") else None
        if "__complex__" in v: return complex(*v["__complex__"])
        if "__dict__" in v: return {_dec(k): _dec(x) for k, x in v["__dict__"]}
        return ("<unrecognised value>", str(v)[:80])
    return v

def _enc(v):
    if isinstance(v, tuple): return {"__tuple__": [_enc(x) for x in v]}
    if isinstance(v, list): return [_enc(x) for x in v]
    if isinstance(v, dict): return {"__dict__": [[_enc(k), _enc(x)] for k, x in v.items()]}
    return v

class RemoteFunction:
    """Calls the AI-written function inside a separate process. Use as a context manager."""
    def __init__(self, code, entry, timeout=10):
        self.deadline = _time.monotonic() + timeout
        self.tmp = tempfile.mkdtemp(prefix="cand_")
        self.p = self.err = None
        self.w1 = self.r2 = None
        self.sel = _selectors.DefaultSelector()
        self.buf = b""
        self._closed = False
        try:
            code_path, child_path = _os.path.join(self.tmp, "candidate.py"), _os.path.join(self.tmp, "child.py")
            open(code_path, "w").write(code); open(child_path, "w").write(_CHILD)
            r1, self.w1 = _os.pipe(); self.r2, w2 = _os.pipe()
            env = {"PATH": "/usr/local/bin:/usr/bin:/bin", "HOME": self.tmp, "LANG": "C.UTF-8"}   # reduced, not isolation
            self.err = open(_os.path.join(self.tmp, "stderr.txt"), "w")
            self.p = subprocess.Popen([sys.executable, "-I", child_path, str(r1), str(w2), code_path, entry],
                                      pass_fds=(r1, w2), stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL,
                                      stderr=self.err, env=env, cwd=self.tmp, start_new_session=True)
            _os.close(r1); _os.close(w2)
            self.sel.register(self.r2, _selectors.EVENT_READ)
            msg = self._read()
            if "load_error" in msg:
                raise CandidateError(msg["load_error"])
            if not msg.get("ready"):
                raise CandidateError("protocol violation at start-up")
        except BaseException:
            self.close()        # kill the child, close fds, delete the scratch dir, on ANY failure
            raise

    def _read(self):
        while b"\n" not in self.buf:
            left = self.deadline - _time.monotonic()
            if left <= 0 or not self.sel.select(timeout=left):
                raise CandidateError("timeout")
            chunk = _os.read(self.r2, 65536)
            if chunk:
                self.buf += chunk
                if len(self.buf) > 4_000_000:            # 4 MB cap on one response line
                    raise CandidateError("protocol violation (oversized response)")
                continue
            if not chunk:
                tail = open(self.err.name).read().strip().splitlines()
                raise CandidateError("process ended" + (f": {tail[-1][:120]}" if tail else ""))
        line, self.buf = self.buf.split(b"\n", 1)
        try:
            return json.loads(line)
        except ValueError:
            raise CandidateError("protocol violation")
    def __call__(self, *args, **kwargs):
        rid = _secrets.token_hex(8)
        try:
            _os.write(self.w1, (json.dumps({"id": rid, "args": _enc(list(args)), "kwargs": _enc(kwargs)}) + "\n").encode())
        except OSError:
            raise CandidateError("process ended")
        msg = self._read()
        if msg.get("id") != rid:
            raise CandidateError("protocol violation")
        if "error" in msg:
            raise CandidateError(msg["error"])
        return _dec(msg.get("value"))
    def close(self):
        if self._closed:
            return
        self._closed = True
        if self.p is not None:
            for _ in range(50):                      # SIGKILL can't interrupt a syscall mid-import; wait briefly
                try:
                    _os.killpg(self.p.pid, _signal.SIGKILL)
                except (ProcessLookupError, PermissionError, OSError):
                    break
                try:
                    if self.p.wait(timeout=0.1) is not None:
                        break
                except subprocess.TimeoutExpired:
                    continue
                except Exception:
                    break
        try:
            self.sel.close()
        except Exception:
            pass
        for fd in (self.w1, self.r2):
            if fd is not None:
                try:
                    _os.close(fd)
                except OSError:
                    pass
        if self.err is not None:
            try:
                self.err.close()
            except Exception:
                pass
        import shutil as _shutil
        _shutil.rmtree(self.tmp, ignore_errors=True)       # remove the scratch directory

    def __enter__(self):
        return self
    def __exit__(self, *exc):
        self.close()

_MAX_ABS = 1e308          # anything bigger (or a huge int) is treated as an invalid return, never compared

def _close_enough(v, e, rel_tol, abs_tol=0.0):
    """Decided here, with this process's own arithmetic. Rejects wrong TYPES, non-finite and
    oversized values rather than raising, so a hostile return cannot crash the notebook."""
    if isinstance(e, list):
        return isinstance(v, (list, tuple)) and len(v) == len(e) and all(_close_enough(a, b, rel_tol, abs_tol) for a, b in zip(v, e))
    if isinstance(v, bool) or not isinstance(v, (int, float)):
        return False
    try:
        if isinstance(v, int) and abs(v) > _MAX_ABS:       # avoid int->float OverflowError
            return False
        if not math.isfinite(v) or abs(v) > _MAX_ABS:
            return False
        return abs(v - e) <= max(rel_tol * abs(e), abs_tol)
    except (OverflowError, ValueError, ArithmeticError):
        return False

def _run_tests(f, tests):
    out = []
    for t in tests:
        try:
            v = f(*t["args"])
            out.append({"ok": _close_enough(v, t["expected"], t["rel_tol"], t.get("abs_tol", 0.0)), "value": v})
        except CandidateError as e:
            if str(e).startswith(("timeout", "process ended", "protocol")):
                raise
            out.append({"ok": False, "value": None, "error": str(e)[:120]})
    return out

def evaluate(code, problem, timeout=10):
    """Grade AI-written code: it runs in its own process; tests, held-out cases and sanity checks
    are evaluated HERE. Pass a problem copy with "tests": [] (checks only) or "sanity_checks": []
    (tests only)."""
    r = {"tests": [], "holdout": [], "checks": [], "error": None}
    try:
        with RemoteFunction(PACK.get("imports", "") + "\n" + code, problem["entry_point"], timeout) as f:
            r["tests"] = _run_tests(f, problem["tests"])
            r["holdout"] = _run_tests(f, problem.get("holdout_tests", []))
            for c in problem.get("sanity_checks", []):
                try:
                    exec(c["code"], {"f": f, "math": math})
                    r["checks"].append(True)
                except CandidateError as e:
                    if str(e).startswith(("timeout", "process ended", "protocol")):
                        raise
                    r["checks"].append(False)
                except Exception:
                    r["checks"].append(False)
    except CandidateError as e:
        r["error"] = str(e)[:160]
        r["tests"] = [dict(t, ok=False) for t in r["tests"]] or []
        r["checks"] = [False] * len(problem.get("sanity_checks", [])) if problem.get("sanity_checks") else []
    r["passed"] = bool(r["tests"]) and all(t["ok"] for t in r["tests"]) and r["error"] is None
    r["holdout_passed"] = (all(t["ok"] for t in r["holdout"]) and r["error"] is None) if r["holdout"] else None
    r["checks_passed"] = (all(r["checks"]) and r["error"] is None) if r["checks"] else None
    return r

def is_evidence(source):
    """Only real captures and live calls count as model evidence; fixtures and mismatched captures never do."""
    return str(source).startswith(("capture:", "live:")) and "(prompt changed)" not in str(source)


def wilson(k, n, z=1.96):
    """95% Wilson interval for k successes in n trials."""
    if n == 0:
        return (0.0, 1.0)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def show_grid(rows, cols, cell):
    """Coloured pass/fail grid in Colab (pandas Styler); plain table elsewhere."""
    import pandas as pd
    df = pd.DataFrame([[cell(r, c) for c in cols] for r in rows], index=rows, columns=cols)
    try:
        from IPython.display import display
        display(df.style.map(lambda v: "background:#cfe9dd" if v == "pass" else
                             "background:#f2cccc" if v == "FAIL" else ""))
    except Exception:
        print(df.to_string())
    return df

def write_artifacts(results, extra=None):
    """A1 results.csv + A2 run_manifest.json — every lab writes both."""
    import csv
    keys = sorted({k for r in results for k in r})
    with open("results.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(results)
    manifest = {"pack": PACK.get("pack"), "pack_version": PACK.get("version"),
                "pack_canonical_content_sha256": PACK.get("canonical_content_sha256", PACK.get("sha256")),
                "pack_file_sha256": globals().get("PACK_FILE_SHA256"),
                "requested_temperature": globals().get("TEMPERATURE"),
                "temperature_applied": ("cached samples: " + ", ".join(sorted({str(x.get("temperature") if x.get("temperature") is not None else "provider default") for v in CACHE.values() for x in v.get("samples", [])})))
                                       if USE_CACHE else ("provider default (not controllable)" if str(MODEL).startswith("colab:") else globals().get("TEMPERATURE")),
                "domain": DOMAIN, "model": MODEL, "use_cache": USE_CACHE,
                "cache_models": sorted({s.get("model", "?") for e in CACHE.values() for s in e.get("samples", [])}),
                "run_date": datetime.date.today().isoformat(), "python": platform.python_version()}
    manifest.update(extra or {})
    json.dump(manifest, open("run_manifest.json", "w"), indent=1)
    print("wrote results.csv and run_manifest.json")
    return manifest

print("helpers ready")


**Step 5 — Three prompts.** Ten domain functions, each asked three ways: *minimal*, *full* (a proper spec), and *tests_first* (the spec plus the pack's visible tests). A tests-first answer has seen its tests, so passing them shows little. Every answer is also scored on two **held-out cases** per item that no prompt ever shows — the fair comparison. Only real captures or live calls are scored.

In [ ]:
rows, codes, results = [], {}, {}
for item in ITEMS:
    for v in VARIANTS:
        p, key, prompt = PROBLEMS[item], f"{item}|{v}", make_prompt(PROBLEMS[item], v); log_prompt(key, prompt)
        codes[item, v] = code = extract_code(ask_llm(prompt, key)[0])
        results[item, v] = r = evaluate(code, p)
        r["holdout_cases_passed"] = sum(t["ok"] for t in r["holdout"]); r["holdout_cases_total"] = len(r["holdout"])
        rows.append({"item": item, "variant": v, "passed": r["passed"], "holdout_passed": r["holdout_passed"],
                     "holdout_cases_passed": r["holdout_cases_passed"], "holdout_cases_total": r["holdout_cases_total"],
                     "checks_passed": r["checks_passed"], "error": r.get("error"), "source": provenance(key, prompt)})
report_variants(ITEMS, VARIANTS, rows, results)

**Step 6 — The fix-it loop.** For each minimal-prompt failure, paste the real failure back and ask for a fix — up to three rounds. The table counts repairs *attempted*, *missing from the cache*, and *stale* (a cached repair generated from a different prompt — never used), plus *new passes*. `repairs.csv` records every repair.

In [ ]:
import pandas as pd
by_round, repairs = run_repairs(ITEMS, codes, results)
print(pd.DataFrame(by_round).to_string(index=False))

**Step 7 — Sanity checks.** Each item comes with two rules that must hold, run on the full-prompt answers. Then Hypothesis tries one property on generated inputs; the property is checked here while the AI's function runs in its own process. Passing is evidence, not proof.

In [ ]:
checks = [{"item": i, "test": results[i, "full"]["passed"], **{f"check{k + 1}": c for k, c in enumerate(results[i, "full"]["checks"])}} for i in ITEMS]
show_grid(ITEMS, ["test", "check1", "check2"], lambda i, c: "pass" if next(x for x in checks if x["item"] == i).get(c) else "FAIL")
if PROPERTY:
    print("Hypothesis:", check_property(codes[PROPERTY[0], "full"], *PROPERTY))

## Aha 1 — tests have blind spots
**Step 8.** One item passes its test but fails a sanity check. (If the captured answer doesn't show it, the pack's planted mistake is shown instead, labelled *injected*.)

In [ ]:
item, defect = AHA1
r, source = results[item, "full"], rows[[x["item"] for x in rows].index(item) + 1]["source"]
if not (r["passed"] and r["checks_passed"] is False):
    r, source = evaluate(DEFECTS[defect]["code"], PROBLEMS[item]), f"injected {defect} (planted mistake, not model output)"
print(f"{item} ({source}): test {'passes' if r['passed'] else 'fails'}")
for name, ok in zip([c["name"] for c in PROBLEMS[item]["sanity_checks"]], r["checks"]):
    print(f"   {'pass' if ok else 'FAIL'}  {name}")

## Aha 2 — sanity checks have blind spots
**Step 9.** Plant two known mistakes and run **only the sanity checks** — no reference values yet.

In [ ]:
checks_only = {d: evaluate(DEFECTS[d]["code"], dict(PROBLEMS[DEFECTS[d]["problem"]], tests=[])) for d in BREAKIT}
for d in AHA2:
    print(f"{d} ({DEFECTS[d]['description']}): passes every sanity check? {checks_only[d]['checks_passed']}")
print("(planted mistakes, not model output)")

**Step 10 — Now turn on the reference values.** Run the same planted mistakes again, this time against the pack's reference-value tests only. Then put both detection methods side by side for every planted mistake, against what the pack recorded.

In [ ]:
tests_only = {d: evaluate(DEFECTS[d]["code"], dict(PROBLEMS[DEFECTS[d]["problem"]], sanity_checks=[])) for d in BREAKIT}
for d in AHA2:
    print(f"{d}: reference-value test {'catches it' if not tests_only[d]['passed'] else 'MISSES it'}")
breakit = []
for d in BREAKIT:
    got = {"test": "MISSED" if tests_only[d]["passed"] else "caught", "sanity": "MISSED" if checks_only[d]["checks_passed"] else "caught"}
    breakit.append({"defect": d, "item": DEFECTS[d]["problem"], **got, "matches_pack": got == DEFECTS[d]["expected_detection"]})
print(pd.DataFrame(breakit).to_string(index=False))
if not all(b["matches_pack"] for b in breakit):
    print("NOTE: a result differs from the pack's recorded detection - that is a finding worth discussing.")

**Step 11 — Unit-aware input.** A library like Pint converts tagged units and, in strict mode, refuses a bare number. It cannot detect a unit that was declared wrongly, and this demo checks only the input — the outputs stay plain numbers. It uses the pack's trusted solution, so it runs safely in the notebook.

In [ ]:
if UNIT_DEMO:
    try:
        import pint
        item, qty, unit, needs = UNIT_DEMO
        ns = {}
        exec(PACK["imports"] + PROBLEMS[item]["canonical_solution"], ns)
        ureg = pint.UnitRegistry()
        f = ureg.wraps(None, getattr(ureg, needs))(ns[PROBLEMS[item]["entry_point"]])
        print(f"{qty} {unit} ->", f(qty * getattr(ureg, unit)))
        try:
            f(qty)
        except Exception as e:
            print(f"bare number {qty} -> refused: {type(e).__name__}")
    except ImportError:
        print("Pint is not installed here; run Step 1.")

**Step 12 — Write the evidence.** `results.csv` and `run_manifest.json`, plus the prompt comparison, the fix-it rounds and every repair, the check matrix, the break-it matrix, and every prompt sent (`prompts_sent.csv`).

In [ ]:
import csv
def save(name, records):
    with open(name, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=sorted({k for r in records for k in r})); w.writeheader(); w.writerows(records)
manifest = write_artifacts(rows, extra={"lab": "Lab 2", "fixture_cache": FIXTURE})
save("prompt_variants.csv", [{"item": i, **{v: results[i, v]["passed"] for v in VARIANTS}} for i in ITEMS])
save("fixit_rounds.csv", by_round); save("check_matrix.csv", checks); save("breakit_matrix.csv", breakit)
save("repairs.csv", repairs or [{"item": None}]); save("prompts_sent.csv", PROMPT_LOG)
print("files:", [f for f in os.listdir(".") if f.endswith((".csv", ".json")) and f not in (PACK_FILE, CACHE_FILE)])

## Try it on your own
1. Write one sanity check for a function you care about, as a one-line `assert` using `f` for the function.
2. Plant your own mistake in a trusted solution. Which checks notice?
3. Write a sanity check that catches D5 — the quaternion read in the wrong order. (Hint: it has to know the convention.)